In [1]:
"""
Bai06 - Lab thực hành Chương 14: GenAI cho Quản lý Thời gian chạy và Bộ nhớ
Sách: Supercharged Coding with Gen AI

Mục tiêu:
  - Profiling runtime với timeit và time
  - Phân tích Big-O: O(2^n) → O(n) cho Fibonacci
  - Tối ưu hóa bộ nhớ với generator thay vì DataFrame đầy đủ
  - Minh họa chained prompts với GenAI

Prompt mẫu dùng với Copilot/ChatGPT:
  Chain 1: "Profile the runtime of fibonacci_recursive for n=5,10,20,30"
  Chain 2: "Explain why fibonacci_recursive is O(2^n)"
  Chain 3: "Optimize fibonacci_recursive to O(n) time and O(1) space"
"""

import time
import tracemalloc
from functools import lru_cache


# ─────────────────────────────────────────────
# BEFORE: Fibonacci đệ quy – O(2^n) không tối ưu
# ─────────────────────────────────────────────
def fibonacci_recursive(n: int) -> int:
    """Compute nth Fibonacci number using naive recursion.

    Warning: Exponential time complexity O(2^n). Impractical for n > 35.

    Args:
        n (int): Position in Fibonacci sequence (0-indexed).

    Returns:
        int: nth Fibonacci number.
    """
    if n <= 1:
        return n
    return fibonacci_recursive(n - 1) + fibonacci_recursive(n - 2)


# ─────────────────────────────────────────────
# AFTER: Fibonacci với memoization – O(n) time, O(n) space
# GenAI gợi ý dùng @lru_cache hoặc dynamic programming
# ─────────────────────────────────────────────
@lru_cache(maxsize=None)
def fibonacci_memoized(n: int) -> int:
    """Compute nth Fibonacci number using memoization.

    Time: O(n) | Space: O(n) cache

    Args:
        n (int): Position in Fibonacci sequence (0-indexed).

    Returns:
        int: nth Fibonacci number.
    """
    if n <= 1:
        return n
    return fibonacci_memoized(n - 1) + fibonacci_memoized(n - 2)


def fibonacci_iterative(n: int) -> int:
    """Compute nth Fibonacci number using iteration.

    Optimal: Time O(n) | Space O(1) – no recursion stack.

    Args:
        n (int): Position in Fibonacci sequence (0-indexed).

    Returns:
        int: nth Fibonacci number.
    """
    if n <= 1:
        return n
    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b


# ─────────────────────────────────────────────
# Profiling Runtime
# ─────────────────────────────────────────────
def profile_runtime(func, *args) -> tuple[float, any]:
    """Measure execution time of a function.

    Args:
        func: Function to profile.
        *args: Arguments to pass to the function.

    Returns:
        tuple[float, any]: (elapsed_seconds, result)
    """
    start = time.perf_counter()
    result = func(*args)
    elapsed = time.perf_counter() - start
    return elapsed, result


# ─────────────────────────────────────────────
# BEFORE: get_top_video – Tải toàn bộ CSV vào RAM – O(R*C) space
# ─────────────────────────────────────────────
def get_top_video_naive(data: list[list]) -> int:
    """Find the video with highest average watch ratio (memory-inefficient).

    Loads entire matrix into memory – O(rows * cols) space.

    Args:
        data (list[list]): 2D matrix of watch ratios (None = not watched).

    Returns:
        int: Index of the top video (column index).
    """
    if not data or not data[0]:
        return -1

    cols = len(data[0])
    col_sums = [0.0] * cols
    col_counts = [0] * cols

    for row in data:
        for j, val in enumerate(row):
            if val is not None:
                col_sums[j] += val
                col_counts[j] += 1

    avgs = [
        col_sums[j] / col_counts[j] if col_counts[j] > 0 else 0.0
        for j in range(cols)
    ]
    return avgs.index(max(avgs))


# ─────────────────────────────────────────────
# AFTER: get_top_video – Dùng generator – O(cols) space
# GenAI gợi ý streaming/chunked reading để tiết kiệm RAM
# ─────────────────────────────────────────────
def get_top_video_optimized(row_generator) -> int:
    """Find the top video using row-by-row generator (memory-efficient).

    Processes data row-by-row using a generator, keeping only column
    running totals in memory – O(cols) space.

    Args:
        row_generator: Generator yielding rows (list of float | None).

    Returns:
        int: Index of the video with highest average watch ratio.
    """
    col_sums = None
    col_counts = None

    for row in row_generator:
        if col_sums is None:
            col_sums = [0.0] * len(row)
            col_counts = [0] * len(row)
        for j, val in enumerate(row):
            if val is not None:
                col_sums[j] += val
                col_counts[j] += 1

    if col_sums is None:
        return -1

    avgs = [
        col_sums[j] / col_counts[j] if col_counts[j] > 0 else 0.0
        for j in range(len(col_sums))
    ]
    return avgs.index(max(avgs))


def simulate_row_generator(rows: int, cols: int):
    """Simulate a large dataset as a row generator (avoids loading all to RAM).

    Args:
        rows (int): Number of user rows.
        cols (int): Number of video columns.

    Yields:
        list: One row of watch ratios (70% None = not watched).
    """
    import random
    random.seed(42)
    for i in range(rows):
        row = [
            round(random.random(), 2) if random.random() > 0.7 else None
            for _ in range(cols)
        ]
        # Bias: video at index 2 is always watched more
        row[2] = round(0.8 + random.random() * 0.2, 2)
        yield row


# ─────────────────────────────────────────────
# Demo profiling bộ nhớ với tracemalloc
# ─────────────────────────────────────────────
def measure_memory(func, *args) -> tuple[float, any]:
    """Measure peak memory usage of a function call.

    Args:
        func: Function to profile.
        *args: Arguments passed to func.

    Returns:
        tuple[float, any]: (peak_memory_KB, result)
    """
    tracemalloc.start()
    result = func(*args)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak / 1024, result  # Convert bytes to KB


# ─────────────────────────────────────────────
# Demo chạy
# ─────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 55)
    print("Ch14 Lab: GenAI for Runtime and Memory Management")
    print("=" * 55)

    # ── Fibonacci Benchmark ──
    print("\n[1] Fibonacci Runtime Comparison")
    print(f"{'Method':<25} {'n=30':>12} {'n=35':>12}")
    print("-" * 50)

    t, r = profile_runtime(fibonacci_iterative, 35)
    print(f"{'fibonacci_iterative':<25} {'-':>12} {t*1000:.4f}ms  → {r}")

    t, r = profile_runtime(fibonacci_memoized, 35)
    print(f"{'fibonacci_memoized':<25} {'-':>12} {t*1000:.4f}ms  → {r}")

    t, r = profile_runtime(fibonacci_recursive, 30)
    print(f"{'fibonacci_recursive':<25} {t*1000:.2f}ms  {'(n=35 skipped: too slow)':>12}")

    print("\n[Big-O Summary]")
    print("  fibonacci_recursive: O(2^n) – exponential")
    print("  fibonacci_memoized:  O(n)   – linear (space O(n))")
    print("  fibonacci_iterative: O(n)   – linear (space O(1)) ✓ Best")

    # ── Memory Benchmark ──
    print("\n[2] Memory: get_top_video comparison (100 users × 50 videos)")
    ROWS, COLS = 100, 50
    sample_data = list(simulate_row_generator(ROWS, COLS))

    mem_naive, top_naive = measure_memory(get_top_video_naive, sample_data)
    print(f"  Naive  (full matrix load): peak = {mem_naive:.2f} KB → top video: {top_naive}")

    mem_opt, top_opt = measure_memory(
        get_top_video_optimized, simulate_row_generator(ROWS, COLS)
    )
    print(f"  Optimized (generator):     peak = {mem_opt:.2f} KB → top video: {top_opt}")
    print(f"  Memory saved: {mem_naive - mem_opt:.2f} KB ({(1 - mem_opt/mem_naive)*100:.1f}% reduction)")


Ch14 Lab: GenAI for Runtime and Memory Management

[1] Fibonacci Runtime Comparison
Method                            n=30         n=35
--------------------------------------------------
fibonacci_iterative                  - 0.0039ms  → 9227465
fibonacci_memoized                   - 0.0123ms  → 9227465
fibonacci_recursive       130.30ms  (n=35 skipped: too slow)

[Big-O Summary]
  fibonacci_recursive: O(2^n) – exponential
  fibonacci_memoized:  O(n)   – linear (space O(n))
  fibonacci_iterative: O(n)   – linear (space O(1)) ✓ Best

[2] Memory: get_top_video comparison (100 users × 50 videos)
  Naive  (full matrix load): peak = 4.84 KB → top video: 2
  Optimized (generator):     peak = 3.23 KB → top video: 2
  Memory saved: 1.61 KB (33.3% reduction)
